# Produce Your Own Movie — Data Scraping Notebook

Fetches Bollywood/Tollywood/Kollywood movie data from **TMDB API** and exports a CSV for ML model training.

## Before you run

### Get a free TMDB API key (2 minutes)
1. Go to [https://www.themoviedb.org/signup](https://www.themoviedb.org/signup) → create free account
2. Go to **Settings → API** (in your profile menu)
3. Click **Create** → choose **Developer**
4. Fill in the form (app name: "Produce Your Own Movie", personal use)
5. Copy the **API Key (v3 auth)** — it looks like `a1b2c3d4e5f6...`
6. Paste it in Cell 3 below: `TMDB_API_KEY = "your_key_here"`

### What this notebook does
- **Cell 3**: Configure crew lists and settings (edit here to add/remove people)
- **Cells 4–6**: Setup helpers
- **Cell 7**: Find each crew member's TMDB person ID
- **Cell 8**: Collect movies — seed list (from crew images) + expanded (all movies per person)
- **Cell 9**: Fetch full details + cast/crew for every movie
- **Cell 10**: Map cast/crew to our fixed crew lists per role
- **Cells 11–12**: Build, clean, and export `output/bollywood_crew_data.csv`

### Expected runtime
First run: **15–25 minutes** (API calls, rate-limited to ~3 req/sec).  
Subsequent runs: **< 1 minute** (all responses cached in `cache/`).

### Output schema
| Column | Description |
|---|---|
| `tmdb_id` | Unique TMDB movie ID |
| `movie_title` | Movie title |
| `year` | Release year |
| `original_language` | hi / te / ta |
| `director` | Matched director name or empty |
| `hero` | Matched actor name or empty |
| `heroine` | Matched actress name or empty |
| `villain` | Matched villain name or empty |
| `music_composer` | Matched composer name or empty |
| `genres` | Pipe-separated genres, e.g. `Action\|Drama` |
| `tmdb_rating` | TMDB community rating (0–10) |
| `vote_count` | Number of votes (quality signal) |
| `matched_roles` | How many of the 5 roles were matched |
| `source` | `seed` (from images) or `expanded` (per-person credits) |

In [7]:
# Run once to install dependencies
%pip install requests pandas tqdm


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [27]:
# ============================================================
# CONFIGURATION — edit this cell to add/remove crew members
# All crew lists live here. No names are hardcoded elsewhere.
# ============================================================

# --- TMDB API KEY (required) ---
TMDB_API_KEY = "d1b8c84a8e1603f1053f7860e67e087f"   # <-- paste your key here


# --- CREW LISTS ---

DIRECTORS = [
    "Yash Chopra",
    "Sanjay Leela Bhansali",
    "Rajkumar Hirani",
    "Karan Johar",
    "Aditya Chopra",
    "Rohit Shetty",
    "Farah Khan",
    "Anurag Basu",
    "Imtiaz Ali",
    "Zoya Akhtar",
    "Ashutosh Gowariker",
    "Sandeep Reddy Vanga",
    "Ayan Mukerji",
    "Anurag Kashyap",
    "Amar Kaushik",
    "Atlee",
    "Aditya Dhar",
    "S. S. Rajamouli",
]

ACTORS = [
    "Ranbir Kapoor",
    "Ranveer Singh",
    "Vicky Kaushal",
    "Shahid Kapoor",
    "Kartik Aaryan",
    "Varun Dhawan",
    "Ayushmann Khurrana",
    "Sidharth Malhotra",
    "Emraan Hashmi",
    "R. Madhavan",
    "Riteish Deshmukh",
    "Sushant Singh Rajput",
    "Shah Rukh Khan",
    "Salman Khan",
    "Aamir Khan",
    "Akshay Kumar",
    "Ajay Devgn",
    "Hrithik Roshan",
    "Sanjay Dutt",
    "Saif Ali Khan",
    "Prabhas",
    "Allu Arjun",
    "Ram Charan",
    "Jr. NTR",
    "Mahesh Babu",
]

ACTRESSES = [
    "Deepika Padukone",
    "Alia Bhatt",
    "Kareena Kapoor Khan",
    "Katrina Kaif",
    "Priyanka Chopra Jonas",
    "Shraddha Kapoor",
    "Kiara Advani",
    "Kriti Sanon",
    "Rashmika Mandanna",
    "Triptii Dimri",
    "Madhuri Dixit",
    "Kajol",
    "Aishwarya Rai Bachchan",
    "Yami Gautam",
    "Mrunal Thakur",
    "Sharvari",
    "Disha Patani",
    "Rani Mukerji",
    "Preity Zinta",
    "Karisma Kapoor",
    "Anushka Sharma",
    "Janhvi Kapoor",
    "Sara Ali Khan",
    "Kangana Ranaut",
    "Ananya Panday",
]

VILLAINS = [
    "Amrish Puri",
    "Arjun Kapoor",
    "Rana Daggubati",
    "Vidya Balan",
    "Tabu",
    "John Abraham",
    "Saif Ali Khan",
    "Prakash Raj",
    "Urmila Matondkar",
    "Akshaye Khanna",
    "Vijay Sethupathi",
    "Pankaj Tripathi",
    "Bobby Deol",
    "Arjun Rampal",
    "Raghav Juyal",
    "Richa Chadha",
    "Ranjeet",
    "Nawazuddin Siddiqui",
    "Sonu Sood",
    "Danny Denzongpa",
    "Jackie Shroff",
    "Sushmita Sen",
]

COMPOSERS = [
    "A. R. Rahman",
    "Pritam",
    "Shankar-Ehsaan-Loy",
    "Vishal-Shekhar",
    "Amit Trivedi",
    "Anu Malik",
    "Jatin-Lalit",
    "Bappi Lahiri",
    "Rajesh Roshan",
    "Sachin-Jigar",
    "Ajay-Atul",
    "Amaal Mallik",
    "Tanishk Bagchi",
    "Mithoon",
    "Salim-Sulaiman",
    "Shantanu Moitra",
    "Ram Sampath",
    "Jeet Gannguli",
    "Sachet-Parampara",
    "Darshan Raval",
    "Himesh Reshammiya",
    "Jasleen Royal",
]

# Duo/trio composers: canonical name → individual TMDB-searchable names.
# When any member has a "Music" credit in a movie, the canonical name is assigned.
COMPOSER_ALIASES = {
    "Shankar-Ehsaan-Loy":  ["Shankar Mahadevan", "Ehsaan Noorani", "Loy Mendonsa"],
    "Vishal-Shekhar":      ["Vishal Dadlani", "Shekhar Ravjiani"],
    "Jatin-Lalit":         ["Jatin Pandit", "Lalit Pandit"],
    "Sachin-Jigar":        ["Sachin Sanghvi", "Jigar Saraiya"],
    "Ajay-Atul":           ["Ajay Gogavale", "Atul Gogavale"],
    "Salim-Sulaiman":      ["Salim Merchant", "Sulaiman Merchant"],
    "Sachet-Parampara":    ["Sachet Tandon", "Parampara Thakur"],
}


# --- SEED MOVIES (from the crew images) ---
# Format: ("Title", year)
# These are fetched first, then expanded via per-person credits.

SEED_MOVIES = [
    # Directors
    ("Veer-Zaara", 2004),
    ("Dil To Pagal Hai", 1997),
    ("Darr", 1993),
    ("Devdas", 2002),
    ("Black", 2005),
    ("Bajirao Mastani", 2015),
    ("3 Idiots", 2009),
    ("Munna Bhai M.B.B.S.", 2003),
    ("PK", 2014),
    ("Kuch Kuch Hota Hai", 1998),
    ("Kabhi Khushi Kabhie Gham", 2001),
    ("Rocky Aur Rani Kii Prem Kahaani", 2023),
    ("Dilwale Dulhania Le Jayenge", 1995),
    ("Mohabbatein", 2000),
    ("Rab Ne Bana Di Jodi", 2008),
    ("Chennai Express", 2013),
    ("Singham", 2011),
    ("Golmaal 3", 2010),
    ("Main Hoon Na", 2004),
    ("Om Shanti Om", 2007),
    ("Happy New Year", 2014),
    ("Barfi!", 2012),
    ("Life in a... Metro", 2007),
    ("Gangster", 2006),
    ("Jab We Met", 2007),
    ("Rockstar", 2011),
    ("Tamasha", 2015),
    ("Zindagi Na Milegi Dobara", 2011),
    ("Gully Boy", 2019),
    ("Dil Dhadakne Do", 2015),
    ("Lagaan", 2001),
    ("Swades", 2004),
    ("Jodhaa Akbar", 2008),
    ("Animal", 2023),
    ("Kabir Singh", 2019),
    ("Arjun Reddy", 2017),
    ("Yeh Jawaani Hai Deewani", 2013),
    ("Wake Up Sid", 2009),
    ("Brahmastra: Part One - Shiva", 2022),
    ("Gangs of Wasseypur", 2012),
    ("Dev.D", 2009),
    ("Black Friday", 2007),
    ("Stree", 2018),
    ("Bala", 2019),
    ("Stree 2", 2024),
    ("Jawan", 2023),
    ("Theri", 2016),
    ("Raja Rani", 2013),
    ("Uri: The Surgical Strike", 2019),
    ("Baahubali 2: The Conclusion", 2017),
    ("RRR", 2022),
    ("Baahubali: The Beginning", 2015),
    # Actors
    ("Masaan", 2015),
    ("Sardar Udham", 2021),
    ("Haider", 2014),
    ("Bhool Bhulaiyaa 2", 2022),
    ("Sonu Ke Titu Ki Sweety", 2018),
    ("Satyaprem Ki Katha", 2023),
    ("Badlapur", 2015),
    ("October", 2018),
    ("Humpty Sharma Ki Dulhania", 2014),
    ("Andhadhun", 2018),
    ("Article 15", 2019),
    ("Dream Girl", 2019),
    ("Shershaah", 2021),
    ("Ek Villain", 2014),
    ("Hasee Toh Phasee", 2014),
    ("The Dirty Picture", 2011),
    ("Jannat", 2008),
    ("Once Upon a Time in Mumbaai", 2010),
    ("Rocketry: The Nambi Effect", 2022),
    ("Rang De Basanti", 2006),
    ("Masti", 2004),
    ("Chhichhore", 2019),
    ("MS Dhoni: The Untold Story", 2016),
    ("Kai Po Che!", 2013),
    ("My Name Is Khan", 2010),
    ("Bajrangi Bhaijaan", 2015),
    ("Hum Aapke Hain Koun..!", 1994),
    ("Sultan", 2016),
    ("Dangal", 2016),
    ("Hera Pheri", 2000),
    ("Special 26", 2013),
    ("Airlift", 2016),
    ("Drishyam", 2015),
    ("The Legend of Bhagat Singh", 2002),
    ("Kaho Naa... Pyaar Hai", 2000),
    ("Lakshya", 2004),
    ("Vaastav", 1999),
    ("Khalnayak", 1993),
    ("Dil Chahta Hai", 2001),
    ("Omkara", 2006),
    ("Tanhaji: The Unsung Warrior", 2020),
    ("Salaar: Part 1 - Ceasefire", 2023),
    ("Pushpa: The Rise", 2021),
    ("Pushpa 2: The Rule", 2024),
    ("Ala Vaikunthapurramuloo", 2020),
    ("Rangasthalam", 2018),
    ("Magadheera", 2009),
    ("Jai Lava Kusa", 2017),
    ("Aravinda Sametha Veera Raghava", 2019),
    ("Pokiri", 2006),
    ("Srimanthudu", 2015),
    ("Okkadu", 2003),
    # Actresses
    ("Piku", 2015),
    ("Padmaavat", 2018),
    ("Gangubai Kathiawadi", 2022),
    ("Raazi", 2018),
    ("Rajneeti", 2010),
    ("Tiger Zinda Hai", 2017),
    ("Fashion", 2008),
    ("Mary Kom", 2014),
    ("Aashiqui 2", 2013),
    ("Good Newwz", 2019),
    ("Mimi", 2021),
    ("Bareilly Ki Barfi", 2017),
    ("Luka Chuppi", 2019),
    ("Geetha Govindam", 2018),
    ("Sita Ramam", 2022),
    ("Laila Majnu", 2018),
    ("Bulbbul", 2020),
    ("Hum Dil De Chuke Sanam", 1999),
    ("Fanaa", 2006),
    ("Dhoom 2", 2006),
    ("Vicky Donor", 2012),
    ("Article 370", 2024),
    ("Hi Nanna", 2023),
    ("Super 30", 2019),
    ("Munjya", 2024),
    ("Maharaj", 2024),
    ("Vedaa", 2024),
    ("Malang", 2020),
    ("Baaghi 2", 2018),
    ("Hichki", 2018),
    ("Mardaani", 2014),
    ("Kal Ho Naa Ho", 2003),
    ("Raja Hindustani", 1996),
    ("Zubeidaa", 2001),
    ("Band Baaja Baaraat", 2010),
    ("NH10", 2015),
    ("Dhadak", 2018),
    ("Good Luck Jerry", 2022),
    ("Mili", 2022),
    ("Kedarnath", 2018),
    ("Atrangi Re", 2021),
    ("Zara Hatke Zara Bachke", 2023),
    ("Queen", 2014),
    ("Tanu Weds Manu", 2011),
    ("Kho Gaye Hum Kahan", 2023),
    ("CTRL", 2024),
    ("Dream Girl 2", 2023),
    # Villains
    ("Mr. India", 1987),
    ("Karan Arjun", 1995),
    ("Ghayal", 1990),
    ("Singham Again", 2024),
    ("Kuttey", 2023),
    ("Aurangzeb", 2013),
    ("The Ghazi Attack", 2017),
    ("Bhool Bhulaiyaa 3", 2024),
    ("Kahaani 2", 2016),
    ("Maqbool", 2003),
    ("Dhoom", 2004),
    ("Pathaan", 2023),
    ("Race 2", 2013),
    ("Wanted", 2009),
    ("Ghilli", 2004),
    ("Kaun", 1999),
    ("Humraaz", 2002),
    ("Race", 2008),
    ("Vikram Vedha", 2017),
    ("Master", 2021),
    ("Agneepath", 2012),
    ("Raavan", 2010),
    ("Race 3", 2018),
    ("Love Hostel", 2022),
    ("Ra.One", 2011),
    ("Dhaakad", 2022),
    ("Kill", 2024),
    ("Kisi Ka Bhai Kisi Ki Jaan", 2023),
    ("Street Dancer 3D", 2020),
    ("Fukrey Returns", 2017),
    ("Kick", 2014),
    ("Raman Raghav 2.0", 2016),
    ("Dabangg", 2010),
    ("Simmba", 2018),
    ("Yuva", 2004),
    ("Ghatak", 1996),
    ("Krantiveer", 1994),
    ("Dhoom 3", 2013),
    ("Mission Kashmir", 2000),
    ("Adipurush", 2023),
    # Composers
    ("Dil Se..", 1998),
    ("Taal", 1999),
    ("Ae Dil Hai Mushkil", 2016),
    ("Rock On!!", 2008),
    ("Student of the Year", 2012),
    ("Ishaqzaade", 2012),
    ("Baazigar", 1993),
    ("Fiza", 2000),
    ("Bhediya", 2022),
    ("Sairat", 2016),
    ("Roy", 2015),
    ("Badrinath Ki Dulhania", 2017),
    ("JugJugg Jeeyo", 2022),
    ("Jalebi", 2019),
    ("Chak De! India", 2007),
    ("Ishqiya", 2010),
    ("Parineeta", 2005),
    ("Delhi Belly", 2011),
    ("Talaash", 2012),
    ("Hamari Adhuri Kahani", 2015),
    ("Khamoshiyan", 2015),
    ("Jersey", 2022),
    ("Ek Villain Returns", 2022),
    ("Bell Bottom", 2021),
    ("Haseen Dillruba", 2021),
    ("Laapataa Ladies", 2024),
    ("Disco Dancer", 1982),
    ("Namak Halaal", 1982),
]


# --- FILTERS ---
ALLOWED_LANGUAGES = {"hi", "te", "ta"}   # Hindi, Telugu, Tamil
MIN_YEAR         = 1980

# NOTE: Bollywood/Tollywood films have very few TMDB votes — Indian audiences
# primarily rate on IMDb. Famous films: Animal=211, Kabir Singh=191, Sultan=274.
# 25 is the right floor: filters truly obscure unknowns, keeps all curated films.
MIN_VOTE_COUNT   = 25

MIN_MATCHED_ROLES = 2     # drop rows where fewer than 2 of 5 roles matched

# How deep to search cast list for each role
MAX_CAST_HERO    = 8    # top-N male cast searched for hero
MAX_CAST_HEROINE = 8    # top-N female cast searched for heroine
MAX_CAST_VILLAIN = 15   # villains appear lower in billing, search wider

# Rate limiting (TMDB allows ~40 req/10s; 0.3s gap = ~3 req/s, safely under limit)
REQUEST_DELAY = 0.3

In [28]:
import requests
import pandas as pd
import json
import re
import time
import hashlib
from pathlib import Path
from tqdm.auto import tqdm

assert TMDB_API_KEY != "YOUR_TMDB_API_KEY_HERE", (
    "Paste your TMDB API key in Cell 3 before running."
)

CACHE_DIR  = Path("cache")
OUTPUT_DIR = Path("output")
CACHE_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

TMDB_BASE_URL = "https://api.themoviedb.org/3"

print("Setup complete.")

Setup complete.


In [29]:
# ── TMDB API HELPERS ─────────────────────────────────────────

_session = requests.Session()
_session.headers.update({"Accept": "application/json"})

def _cache_path(endpoint: str, params: dict) -> Path:
    key = endpoint + json.dumps({k: v for k, v in sorted(params.items()) if k != "api_key"})
    return CACHE_DIR / (hashlib.md5(key.encode()).hexdigest() + ".json")


def tmdb_get(endpoint: str, params: dict = None, _retries: int = 5) -> dict:
    """Cached, rate-limited TMDB GET with exponential backoff on transient errors."""
    params = dict(params or {})
    params["api_key"] = TMDB_API_KEY

    cache_file = _cache_path(endpoint, params)
    if cache_file.exists():
        with open(cache_file) as f:
            return json.load(f)

    time.sleep(REQUEST_DELAY)

    for attempt in range(_retries):
        try:
            resp = _session.get(f"{TMDB_BASE_URL}{endpoint}", params=params, timeout=15)
            if resp.status_code == 429:
                wait = 2 ** attempt * 5
                print(f"  Rate limited — waiting {wait}s...")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            data = resp.json()
            with open(cache_file, "w") as f:
                json.dump(data, f)
            return data
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as e:
            if attempt == _retries - 1:
                raise
            wait = 2 ** attempt * 2
            print(f"  Connection error ({e.__class__.__name__}), retry {attempt+1}/{_retries} in {wait}s...")
            time.sleep(wait)
            _session.close()
            _session.__init__()
            _session.headers.update({"Accept": "application/json"})


def search_person(name: str) -> int | None:
    data = tmdb_get("/search/person", {"query": name})
    results = data.get("results", [])
    if not results:
        return None
    return max(results, key=lambda x: x.get("popularity", 0))["id"]


def get_person_movie_credits(person_id: int) -> list:
    data = tmdb_get(f"/person/{person_id}/combined_credits")
    all_entries = data.get("cast", []) + data.get("crew", [])
    seen, movies = set(), []
    for entry in all_entries:
        mid = entry.get("id")
        if entry.get("media_type") == "movie" and mid and mid not in seen:
            seen.add(mid)
            movies.append(entry)
    return movies


def get_movie_details(movie_id: int) -> dict:
    return tmdb_get(f"/movie/{movie_id}")


def get_movie_credits(movie_id: int) -> dict:
    return tmdb_get(f"/movie/{movie_id}/credits")


def search_movie_by_title(title: str, year: int = None) -> int | None:
    params = {"query": title}
    if year:
        params["primary_release_year"] = year
    data = tmdb_get("/search/movie", params)
    results = data.get("results", [])
    for r in results:
        if r.get("original_language") in ALLOWED_LANGUAGES:
            return r["id"]
    return results[0]["id"] if results else None


print("API helpers ready.")

API helpers ready.


In [33]:
# ── NAME NORMALIZATION & LOOKUP SETS ─────────────────────────

def normalize(name: str) -> str:
    """Lowercase, collapse dots/hyphens to spaces, strip extra whitespace."""
    name = name.lower().strip()
    name = re.sub(r"[.\-]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name


def build_lookup(names: list) -> dict:
    """Return {normalized_name: canonical_name} dict."""
    return {normalize(n): n for n in names}


# Primary lookup dicts (normalized → canonical)
dir_lookup      = build_lookup(DIRECTORS)
actor_lookup    = build_lookup(ACTORS)
actress_lookup  = build_lookup(ACTRESSES)
villain_lookup  = build_lookup(VILLAINS)
composer_lookup = build_lookup(COMPOSERS)

# Extended composer lookup: normalized alias member → canonical duo name
composer_alias_lookup = {}
for duo_name, members in COMPOSER_ALIASES.items():
    canonical = duo_name   # e.g. "Shankar-Ehsaan-Loy"
    for member in members:
        composer_alias_lookup[normalize(member)] = canonical

print(f"Lookup sizes — directors: {len(dir_lookup)}, actors: {len(actor_lookup)}, "
      f"actresses: {len(actress_lookup)}, villains: {len(villain_lookup)}, "
      f"composers: {len(composer_lookup)} (+{len(composer_alias_lookup)} aliases)")

Lookup sizes — directors: 18, actors: 25, actresses: 25, villains: 22, composers: 22 (+15 aliases)


In [34]:
# ── BUILD TMDB PERSON ID MAP ─────────────────────────────────
# Finds TMDB person IDs for everyone in the crew lists.
# Alias members (e.g. "Shankar Mahadevan") are looked up individually
# so their movie credits can be fetched for the expanded dataset.

# Flatten all names to search, with role label for diagnostics
_names_to_search = (
    [(n, "director")  for n in DIRECTORS]  +
    [(n, "actor")     for n in ACTORS]     +
    [(n, "actress")   for n in ACTRESSES]  +
    [(n, "villain")   for n in VILLAINS]   +
    [(n, "composer")  for n in COMPOSERS]  +
    [(alias, "composer-alias") for members in COMPOSER_ALIASES.values() for alias in members]
)

PERSON_IDS = {}   # normalized_name → tmdb_person_id
_not_found = []

print(f"Searching TMDB for {len(_names_to_search)} names...")
for name, role in tqdm(_names_to_search):
    norm = normalize(name)
    if norm in PERSON_IDS:
        continue
    pid = search_person(name)
    if pid:
        PERSON_IDS[norm] = pid
    else:
        _not_found.append((name, role))

print(f"\nFound: {len(PERSON_IDS)} / {len(_names_to_search)} names")
if _not_found:
    print(f"\n⚠ Not found on TMDB ({len(_not_found)}) — check spellings or add aliases:")
    for name, role in _not_found:
        print(f"  [{role}]  {name}")

Searching TMDB for 127 names...


100%|██████████| 127/127 [00:00<00:00, 9661.32it/s]


Found: 126 / 127 names


In [35]:
# ── COLLECT ALL MOVIE IDs ─────────────────────────────────────
# Phase A: Seed movies from the crew images (explicit list).
# Phase B: All movies from each person's TMDB credits (expanded).
#
# Result: all_movie_ids = {tmdb_id: "seed" | "expanded"}

all_movie_ids = {}   # tmdb_id → source label

# ── Phase A: Seed movies ──────────────────────────────────────
print("Phase A: Resolving seed movies...")
_seed_not_found = []
for title, year in tqdm(SEED_MOVIES):
    mid = search_movie_by_title(title, year)
    if mid:
        all_movie_ids[mid] = "seed"
    else:
        _seed_not_found.append((title, year))

print(f"  Resolved: {len(all_movie_ids)} / {len(SEED_MOVIES)} seed movies")
if _seed_not_found:
    print(f"  ⚠ Not found: {_seed_not_found}")

# ── Phase B: Per-person expanded credits ──────────────────────
print("\nPhase B: Expanding via per-person credits...")
_before_expand = len(all_movie_ids)

# Only expand for primary role members (not alias duplicates)
_primary_names = (
    DIRECTORS + ACTORS + ACTRESSES + VILLAINS + COMPOSERS +
    [alias for members in COMPOSER_ALIASES.values() for alias in members]
)

for name in tqdm(_primary_names):
    pid = PERSON_IDS.get(normalize(name))
    if not pid:
        continue
    credits = get_person_movie_credits(pid)
    for m in credits:
        mid = m.get("id")
        if not mid or mid in all_movie_ids:
            continue
        # Loose pre-filter (language + year + minimal votes).
        # Strict MIN_VOTE_COUNT applied later in Phase C on full movie details.
        lang  = m.get("original_language", "")
        year  = (m.get("release_date") or "")[:4]
        votes = m.get("vote_count", 0)
        if lang not in ALLOWED_LANGUAGES:
            continue
        if year and int(year) < MIN_YEAR:
            continue
        if votes < 10:   # only reject truly zero-vote entries
            continue
        all_movie_ids[mid] = "expanded"

print(f"  Added {len(all_movie_ids) - _before_expand} movies via expansion")
print(f"  Total unique movies to process: {len(all_movie_ids)}")

Phase A: Resolving seed movies...


100%|██████████| 219/219 [00:00<00:00, 5107.16it/s]


  Resolved: 216 / 219 seed movies
  ⚠ Not found: [('Black Friday', 2007), ('Aravinda Sametha Veera Raghava', 2019), ('Jalebi', 2019)]

Phase B: Expanding via per-person credits...


100%|██████████| 127/127 [00:00<00:00, 2243.82it/s]

  Added 1574 movies via expansion
  Total unique movies to process: 1790


In [36]:
# ── FETCH MOVIE DETAILS + CREDITS ────────────────────────────
# For each unique movie ID, fetch details (genres, rating, language)
# and credits (cast + crew). Applies final vote_count filter.

movie_data = {}   # tmdb_id → {"details": ..., "credits": ..., "source": ...}
_skipped = 0

print(f"Fetching details + credits for {len(all_movie_ids)} movies...")
for movie_id, source in tqdm(all_movie_ids.items()):
    try:
        details = get_movie_details(movie_id)

        # Final filters
        if details.get("original_language") not in ALLOWED_LANGUAGES:
            _skipped += 1
            continue
        if details.get("vote_count", 0) < MIN_VOTE_COUNT:
            _skipped += 1
            continue

        credits = get_movie_credits(movie_id)
        movie_data[movie_id] = {"details": details, "credits": credits, "source": source}

    except Exception as e:
        print(f"  ⚠ Error movie {movie_id}: {e}")
        _skipped += 1

print(f"\nKept: {len(movie_data)} movies  |  Skipped: {_skipped}")

Fetching details + credits for 1790 movies...


100%|██████████| 1790/1790 [24:07<00:00,  1.24it/s]


Kept: 1125 movies  |  Skipped: 665


In [37]:
# ── ROLE MAPPING FUNCTIONS ───────────────────────────────────
# Each function receives raw TMDB cast/crew lists and returns the
# canonical name from our crew lists, or None if no match.

def find_director(crew: list) -> str | None:
    for member in crew:
        if member.get("job") == "Director":
            return dir_lookup.get(normalize(member.get("name", "")))
    return None


def find_hero(cast: list) -> str | None:
    """Top-billed male cast member present in ACTORS list."""
    males = [c for c in cast if c.get("gender") == 2][:MAX_CAST_HERO]
    for member in males:
        canonical = actor_lookup.get(normalize(member.get("name", "")))
        if canonical:
            return canonical
    return None


def find_heroine(cast: list) -> str | None:
    """Top-billed female cast member present in ACTRESSES list."""
    females = [c for c in cast if c.get("gender") == 1][:MAX_CAST_HEROINE]
    for member in females:
        canonical = actress_lookup.get(normalize(member.get("name", "")))
        if canonical:
            return canonical
    return None


def find_villain(cast: list, exclude_normalized: set = None) -> str | None:
    """Match from VILLAINS list across wider cast, excluding already-assigned roles."""
    exclude_normalized = exclude_normalized or set()
    for member in cast[:MAX_CAST_VILLAIN]:
        norm = normalize(member.get("name", ""))
        if norm in exclude_normalized:
            continue
        canonical = villain_lookup.get(norm)
        if canonical:
            return canonical
    return None


def find_composer(crew: list) -> str | None:
    """Match from COMPOSERS list (including duo aliases) via crew credits."""
    music_jobs = {"Original Music Composer", "Music", "Music Composer",
                  "Musician", "Music Director", "Score"}
    for member in crew:
        if member.get("department") == "Sound" or member.get("job") in music_jobs:
            norm = normalize(member.get("name", ""))
            # Direct match
            canonical = composer_lookup.get(norm)
            if canonical:
                return canonical
            # Alias match (individual member of a duo)
            canonical = composer_alias_lookup.get(norm)
            if canonical:
                return canonical
    return None


print("Role mapping functions ready.")

Role mapping functions ready.


In [38]:
# ── BUILD DATASET ─────────────────────────────────────────────

rows = []

for movie_id, data in tqdm(movie_data.items()):
    details = data["details"]
    credits = data["credits"]
    cast    = credits.get("cast", [])
    crew    = credits.get("crew", [])

    director = find_director(crew)
    hero     = find_hero(cast)
    heroine  = find_heroine(cast)

    # Exclude hero/heroine normalized names from villain search
    # (handles Saif Ali Khan appearing in both ACTORS and VILLAINS)
    _exclude = {normalize(x) for x in [hero, heroine] if x}
    villain  = find_villain(cast, _exclude)
    composer = find_composer(crew)

    matched = sum(x is not None for x in [director, hero, heroine, villain, composer])
    if matched < MIN_MATCHED_ROLES:
        continue

    release_date = details.get("release_date") or ""
    year = int(release_date[:4]) if len(release_date) >= 4 else None
    genres = "|".join(g["name"] for g in details.get("genres", []))

    rows.append({
        "tmdb_id":          movie_id,
        "movie_title":      details.get("title", ""),
        "year":             year,
        "original_language": details.get("original_language", ""),
        "director":         director,
        "hero":             hero,
        "heroine":          heroine,
        "villain":          villain,
        "music_composer":   composer,
        "genres":           genres,
        "tmdb_rating":      details.get("vote_average"),
        "vote_count":       details.get("vote_count"),
        "matched_roles":    matched,
        "source":           data["source"],
    })

df = pd.DataFrame(rows)
print(f"Raw dataset: {len(df)} rows, avg {df['matched_roles'].mean():.2f} roles matched per movie")

100%|██████████| 1125/1125 [00:00<00:00, 24321.64it/s]

Raw dataset: 675 rows, avg 2.69 roles matched per movie


In [39]:
# ── CLEAN & EXPORT ────────────────────────────────────────────

# Remove duplicates (same movie from multiple person lookups)
df = df.drop_duplicates(subset=["tmdb_id"]).copy()

# Sort by year then title
df = df.sort_values(["year", "movie_title"]).reset_index(drop=True)

# Fill rate report
role_cols = ["director", "hero", "heroine", "villain", "music_composer"]
print("Fill rates per role (matched to our crew lists):")
for col in role_cols:
    n = df[col].notna().sum()
    pct = 100 * n / len(df)
    bar = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
    print(f"  {col:<20} {bar}  {n}/{len(df)}  ({pct:.1f}%)")

# Export
output_path = OUTPUT_DIR / "bollywood_crew_data.csv"
df.to_csv(output_path, index=False)

print(f"\n✓ Saved → {output_path}")
print(f"  {len(df)} rows × {len(df.columns)} columns")

Fill rates per role (matched to our crew lists):
  director             ███░░░░░░░░░░░░░░░░░  111/675  (16.4%)
  hero                 ███████████████░░░░░  530/675  (78.5%)
  heroine              ███████████░░░░░░░░░  373/675  (55.3%)
  villain              █████████░░░░░░░░░░░  319/675  (47.3%)
  music_composer       ██████████████░░░░░░  480/675  (71.1%)

✓ Saved → output/bollywood_crew_data.csv
  675 rows × 14 columns


In [40]:
# ── DATASET SUMMARY ───────────────────────────────────────────

print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)
print(f"Total movies:         {len(df)}")
print(f"  From seed list:     {(df.source == 'seed').sum()}")
print(f"  From expansion:     {(df.source == 'expanded').sum()}")
print(f"\nYear range:           {int(df.year.min())} – {int(df.year.max())}")
print(f"Rating range:         {df.tmdb_rating.min():.1f} – {df.tmdb_rating.max():.1f}")
print(f"Rating mean / median: {df.tmdb_rating.mean():.2f} / {df.tmdb_rating.median():.2f}")

print("\nLanguage breakdown:")
print(df.original_language.value_counts().to_string())

print("\nTop 10 genres (across all movies):")
genre_counts = df.genres.str.split("|").explode().value_counts()
print(genre_counts.head(10).to_string())

print("\nDirector appearances (top 10):")
print(df.director.value_counts().head(10).to_string())

print("\nSample rows:")
display(df[["movie_title", "year", "director", "hero", "heroine", "villain",
            "music_composer", "tmdb_rating", "matched_roles"]].head(15))

DATASET SUMMARY
Total movies:         675
  From seed list:     157
  From expansion:     518

Year range:           1989 – 2026
Rating range:         2.3 – 8.5
Rating mean / median: 6.21 / 6.30

Language breakdown:
original_language
hi    594
te     52
ta     29

Top 10 genres (across all movies):
genres
Drama        408
Comedy       265
Romance      263
Action       258
Thriller     162
Crime        117
Adventure     46
History       42
Family        37
Horror        25

Director appearances (top 10):
director
Rohit Shetty             14
Anurag Kashyap           13
Sanjay Leela Bhansali     9
Imtiaz Ali                9
Karan Johar               7
Anurag Basu               7
S. S. Rajamouli           7
Ashutosh Gowariker        6
Rajkumar Hirani           6
Zoya Akhtar               5

Sample rows:


,movie_title,year,director,hero,heroine,villain,music_composer,tmdb_rating,matched_roles
0,Parinda,1989,NaN,NaN,Madhuri Dixit,Jackie Shroff,NaN,7.2,2
1,Dil,1990,NaN,Aamir Khan,Madhuri Dixit,NaN,NaN,6.5,2
2,Saajan,1991,NaN,Sanjay Dutt,Madhuri Dixit,NaN,NaN,6.5,2
3,Chamatkar,1992,NaN,Shah Rukh Khan,NaN,Urmila Matondkar,Anu Malik,6.5,3
4,Deewana,1992,NaN,Shah Rukh Khan,NaN,Amrish Puri,NaN,6.8,2
5,Jo Jeeta Wohi Sikandar,1992,NaN,Aamir Khan,NaN,NaN,Jatin-Lalit,7.2,2
6,Khiladi,1992,NaN,Akshay Kumar,NaN,NaN,Jatin-Lalit,6.2,2
7,Raju Ban Gaya Gentleman,1992,NaN,Shah Rukh Khan,NaN,NaN,Salim-Sulaiman,7.1,2
8,Baazigar,1993,NaN,Shah Rukh Khan,Kajol,NaN,Anu Malik,6.9,3
9,Darr,1993,Yash Chopra,Shah Rukh Khan,NaN,NaN,NaN,6.9,2
